Import CollegeBasketballData and other needed requirements. 

In [33]:
import pandas as pd
import requests
import notebook
import sqlalchemy
import psycopg2
import json
import numpy as np
from datetime import datetime
import cbbd
import glob 
import os 
from datetime import datetime, timedelta
 

Pull key froms json

In [4]:
with open("keys.json", "r") as f:
    keys = json.load(f)
    API_KEY = keys.get("API_KEY")


  

Wins Above Bubble Calculation

Calculate WAB for all seasons

In [37]:
season = 2021
bubble_percentile = 0.15
configuration = cbbd.Configuration(
    access_token= API_KEY
)
with cbbd.ApiClient(configuration) as api_client:
    games_raw_1 = cbbd.GamesApi(api_client).get_games(
        start_date_range=datetime(2020, 11, 1),
        end_date_range=datetime(2020, 12, 31),
        status=cbbd.GameStatus.FINAL
    )
    games_raw_2 = cbbd.GamesApi(api_client).get_games(
        start_date_range=datetime(2021, 1, 1),
        end_date_range=datetime(2021, 2, 28),
        status=cbbd.GameStatus.FINAL
    )
    games_raw_3 = cbbd.GamesApi(api_client).get_games(
        start_date_range=datetime(2021, 3, 1),
        end_date_range=datetime(2021, 4, 30),
        status=cbbd.GameStatus.FINAL
    )

    elo_raw = cbbd.RatingsApi(api_client).get_elo(season=season)

games_raw = games_raw_1 + games_raw_2 + games_raw_3




    
games_list = []
for game in games_raw:
    games_list.append({
        'homeTeam': game.home_team,
        'awayTeam': game.away_team,
        'homeScore': game.home_points,
        'awayScore': game.away_points,
    })
games = pd.DataFrame(games_list).drop_duplicates().reset_index(drop=True)
games["homeScore"] = pd.to_numeric(games["homeScore"], errors="coerce")
games["awayScore"] = pd.to_numeric(games["awayScore"], errors="coerce")



elo = pd.DataFrame([rating.to_dict() for rating in elo_raw])    


print("Games shape just before loop:", games.shape)
print("Columns:", games.columns.tolist())
if "status" in games.columns:
   games = games[games["status"].str.contains("FINAL", na=False)].copy()



elo_sorted = elo.sort_values("elo", ascending=False).reset_index(drop=True)
bubble_index = min(int(len(elo_sorted) * bubble_percentile), len(elo_sorted) - 1)
bubble_elo = elo_sorted.iloc[bubble_index]["elo"]

d1_teams = set(elo_sorted["team"])
games = games[
    (games["homeTeam"].isin(d1_teams)) & 
    (games["awayTeam"].isin(d1_teams))
].reset_index(drop=True)

print(f"Season : {season}")
print(f"Teams  : {len(elo_sorted)}")
print(f"Bubble ELO threshold (top {int(bubble_percentile*100)}%): {bubble_elo:.1f}")

elo_dict = dict(zip(elo_sorted['team'], elo_sorted['elo']))
rank_lookup = {row['team']: i+1 for i, row in elo_sorted.iterrows()}

def win_prob_vs_bubble(oppenent_elo : float, bubble_elo: float) -> float:
   diff = oppenent_elo - bubble_elo
   prob = 1.0/(1.0 + 10 ** (diff / 400))
   return float(np.clip(prob, 0.05, 0.95))

all_teams = set(games["homeTeam"]) | set(games["awayTeam"])
results = []

for team in all_teams:  
    team_games = games[
        (games["homeTeam"] == team) | (games["awayTeam"] == team)
    ]
    
    actual_wins = 0
    expected_wins = 0.0
    games_played = 0
    
    for _, game in team_games.iterrows():
        if game["homeTeam"] == team:
            opponent  = game["awayTeam"]
            my_score  = game["homeScore"]
            opp_score = game["awayScore"]
        else:
            opponent  = game["homeTeam"]
            my_score  = game["awayScore"]
            opp_score = game["homeScore"]
        
        if pd.isna(my_score) or pd.isna(opp_score):
            continue
        
        if my_score > opp_score:
            actual_wins += 1
        
        opp_elo = elo_dict.get(opponent, 1500)
        expected_wins += win_prob_vs_bubble(opp_elo, bubble_elo)
        games_played += 1
    
    print(f"{team}: games_played={games_played}, actual_wins={actual_wins}")
    
    if games_played == 0:
        continue
    
    results.append({
        "team": team,
        "wab": round(actual_wins - expected_wins, 3),
        "actual_wins": actual_wins,
        "expected_wins": round(expected_wins, 3),
        "games_played": games_played,
    })

results_df = (
   pd.DataFrame(results)
   .sort_values("wab", ascending=False)
   .reset_index(drop=True)
)
results_df.insert(0, "rank", results_df.index + 1)

print("\nTop 25 - Wins Above Bubble\n")
print(results_df.head(25).to_string(index=False))




   
   




Games shape just before loop: (4215, 4)
Columns: ['homeTeam', 'awayTeam', 'homeScore', 'awayScore']
Season : 2021
Teams  : 347
Bubble ELO threshold (top 15%): 1890.0
South Dakota: games_played=23, actual_wins=13
Utah: games_played=24, actual_wins=11
William & Mary: games_played=17, actual_wins=7
Oklahoma: games_played=27, actual_wins=16
Eastern Illinois: games_played=26, actual_wins=9
American University: games_played=9, actual_wins=3
UC Davis: games_played=17, actual_wins=9
Michigan State: games_played=27, actual_wins=15
Long Beach State: games_played=17, actual_wins=6
NC State: games_played=23, actual_wins=12
Oakland: games_played=30, actual_wins=12
Western Kentucky: games_played=28, actual_wins=20
Miami (OH): games_played=22, actual_wins=11
Wyoming: games_played=25, actual_wins=14
Northern Arizona: games_played=22, actual_wins=6
Cincinnati: games_played=22, actual_wins=12
Gardner-Webb: games_played=25, actual_wins=10
Tennessee Tech: games_played=25, actual_wins=3
Detroit Mercy: game

Import CSV data to database. 


In [31]:
files = glob.glob("data/NCAA Statistics *.csv")

dfs = []
for f in files:
    temp = pd.read_csv(f)
    temp["year"] = os.path.basename(f).replace("NCAA Statistics ", "").replace(".csv", "")
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

# Save to a new CSV
df = df[["year"] + [col for col in df.columns if col != "year"]]
df.to_csv("data/NCAA Statistics Combined.csv", index=False)

print(f"Done! Combined {len(files)} files with {len(df)} total rows.")
print(df.head())

Done! Combined 8 files with 5024 total rows.
       year           Team     Conference  NET Rank  PrevNET  AvgOppNETRank  \
0  Combined        Gonzaga            WCC         1        1            108   
1  Combined         Kansas         Big 12         2        2              1   
2  Combined         Dayton    Atlantic 10         3        3             89   
3  Combined  San Diego St.  Mountain West         4        4            110   
4  Combined         Baylor         Big 12         5        5             46   

   AvgOppNET    WL Conf.Record Non-ConferenceRecord RoadWL    SOS  NonConfSOS  \
0        146  31-2        17-1                 14-1   10-1  111.0       286.0   
1         67  27-3        17-1                 11-2   10-1    1.0         1.0   
2        130  29-2        18-0                 11-2    9-0   27.0        48.0   
3        150  29-2        19-2                 11-0   11-0   98.0       145.0   
4         97  26-4        15-3                 11-1    9-2   58.0       169

In [38]:
seasons = range(2010, 2026)
bubble_percentile = 0.15
all_results = []

configuration = cbbd.Configuration(access_token=API_KEY)

def win_prob_vs_bubble(opponent_elo: float, bubble_elo: float) -> float:
    diff = opponent_elo - bubble_elo
    prob = 1.0 / (1.0 + 10 ** (diff / 400))
    return float(np.clip(prob, 0.05, 0.95))

for season in seasons:
    print(f"\nFetching season {season}...")
    year1 = season - 1
    year2 = season
    feb_end = datetime(year2, 3, 1) - timedelta(days=1)

    with cbbd.ApiClient(configuration) as api_client:
        games_raw_1 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year1, 11, 1),
            end_date_range=datetime(year1, 12, 31),
            status=cbbd.GameStatus.FINAL
        )
        games_raw_2 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year2, 1, 1),
            end_date_range=feb_end,
            status=cbbd.GameStatus.FINAL
        )
        games_raw_3 = cbbd.GamesApi(api_client).get_games(
            start_date_range=datetime(year2, 3, 1),
            end_date_range=datetime(year2, 4, 30),
            status=cbbd.GameStatus.FINAL
        )
        elo_raw = cbbd.RatingsApi(api_client).get_elo(season=season)

    games_raw = games_raw_1 + games_raw_2 + games_raw_3

    games_list = []
    for game in games_raw:
        games_list.append({
            'homeTeam': game.home_team,
            'awayTeam': game.away_team,
            'homeScore': game.home_points,
            'awayScore': game.away_points,
        })

    games = pd.DataFrame(games_list).drop_duplicates().reset_index(drop=True)
    games["homeScore"] = pd.to_numeric(games["homeScore"], errors="coerce")
    games["awayScore"] = pd.to_numeric(games["awayScore"], errors="coerce")

    elo = pd.DataFrame([rating.to_dict() for rating in elo_raw])
    elo_sorted = elo.sort_values("elo", ascending=False).reset_index(drop=True)
    bubble_index = min(int(len(elo_sorted) * bubble_percentile), len(elo_sorted) - 1)
    bubble_elo = elo_sorted.iloc[bubble_index]["elo"]

    d1_teams = set(elo_sorted["team"])
    games = games[
        (games["homeTeam"].isin(d1_teams)) &
        (games["awayTeam"].isin(d1_teams))
    ].reset_index(drop=True)

    print(f"Season : {season}")
    print(f"Teams  : {len(elo_sorted)}")
    print(f"Games  : {len(games)}")
    print(f"Bubble ELO threshold (top {int(bubble_percentile*100)}%): {bubble_elo:.1f}")

    elo_dict = dict(zip(elo_sorted["team"], elo_sorted["elo"]))
    rank_lookup = {row["team"]: i + 1 for i, row in elo_sorted.iterrows()}

    all_teams = set(games["homeTeam"]) | set(games["awayTeam"])
    results = []

    for team in all_teams:
        team_games = games[
            (games["homeTeam"] == team) | (games["awayTeam"] == team)
        ]

        actual_wins = 0
        expected_wins = 0.0
        games_played = 0

        for _, game in team_games.iterrows():
            if game["homeTeam"] == team:
                opponent  = game["awayTeam"]
                my_score  = game["homeScore"]
                opp_score = game["awayScore"]
            else:
                opponent  = game["homeTeam"]
                my_score  = game["awayScore"]
                opp_score = game["homeScore"]

            if pd.isna(my_score) or pd.isna(opp_score):
                continue

            if my_score > opp_score:
                actual_wins += 1

            opp_elo = elo_dict.get(opponent, 1500)
            expected_wins += win_prob_vs_bubble(opp_elo, bubble_elo)
            games_played += 1

        if games_played == 0:
            continue

        results.append({
            "season":        season,
            "team":          team,
            "wab":           round(actual_wins - expected_wins, 3),
            "actual_wins":   actual_wins,
            "expected_wins": round(expected_wins, 3),
            "games_played":  games_played,
            "elo":           round(elo_dict.get(team, 1500), 1),
            "elo_rank":      rank_lookup.get(team),
        })

    all_results.extend(results)
    print(f"Season {season} done — {len(results)} teams calculated")

# Final combined DataFrame — outside the season loop
final_df = (
    pd.DataFrame(all_results)
    .sort_values(["season", "wab"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nTop 25 per season:\n")
for season in seasons:
    print(f"\n--- {season} ---")
    season_df = final_df[final_df["season"] == season].copy()
    season_df.insert(0, "rank", range(1, len(season_df) + 1))
    print(season_df.head(25).to_string(index=False))


Fetching season 2010...
Season : 2010
Teams  : 335
Games  : 4957
Bubble ELO threshold (top 15%): 1888.0
Season 2010 done — 335 teams calculated

Fetching season 2011...
Season : 2011
Teams  : 346
Games  : 5315
Bubble ELO threshold (top 15%): 1868.0
Season 2011 done — 346 teams calculated

Fetching season 2012...
Season : 2012
Teams  : 350
Games  : 5245
Bubble ELO threshold (top 15%): 1853.0
Season 2012 done — 345 teams calculated

Fetching season 2013...
Season : 2013
Teams  : 349
Games  : 5377
Bubble ELO threshold (top 15%): 1874.0
Season 2013 done — 347 teams calculated

Fetching season 2014...
Season : 2014
Teams  : 351
Games  : 5411
Bubble ELO threshold (top 15%): 1861.0
Season 2014 done — 351 teams calculated

Fetching season 2015...
Season : 2015
Teams  : 351
Games  : 5332
Bubble ELO threshold (top 15%): 1873.0
Season 2015 done — 351 teams calculated

Fetching season 2016...
Season : 2016
Teams  : 351
Games  : 5461
Bubble ELO threshold (top 15%): 1897.0
Season 2016 done — 351 te